In [634]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.model_selection import train_test_split  
from sklearn.model_selection import (
    StratifiedKFold, 
    cross_val_score, 
    cross_validate
)

from sklearn.linear_model import LogisticRegression  

from sklearn.metrics import accuracy_score  


In [635]:
RANDOM_STATE = 777

In [636]:
df = pd.read_csv(
    filepath_or_buffer="gaming_and_mental_health.csv"
)

In [637]:
df.head()

,record_id,age,gender,daily_gaming_hours,game_genre,primary_game,gaming_platform,sleep_hours,sleep_quality,sleep_disruption_frequency,...,continued_despite_problems,eye_strain,back_neck_pain,weight_change_kg,exercise_hours_weekly,social_isolation_score,face_to_face_social_hours_weekly,monthly_game_spending_usd,years_gaming,gaming_addiction_risk_level
0,GD0001,17,Male,11.1,Mobile Games,Clash of Clans,PC,3.7,Very Poor,Sometimes,...,True,True,False,6.8,3.7,7,1.3,383.70,3,Severe
1,GD0002,21,Male,3.0,MOBA,Dota 2,PC,7.2,Fair,Rarely,...,False,False,False,0.4,8.5,2,10.7,46.64,1,Low
2,GD0003,23,Male,7.6,FPS,CS:GO,Multi-platform,4.4,Fair,Often,...,True,False,True,1.8,7.1,5,3.2,100.81,6,Severe
3,GD0004,20,Female,7.2,RPG,Skyrim,Multi-platform,5.1,Fair,Often,...,False,True,True,0.2,5.2,4,9.1,51.60,7,High
4,GD0005,18,Male,6.8,Battle Royale,Apex Legends,PC,3.4,Poor,Never,...,False,False,False,0.5,6.1,4,4.5,32.57,1,Moderate


In [638]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   record_id                         1000 non-null   object 
 1   age                               1000 non-null   int64  
 2   gender                            1000 non-null   object 
 3   daily_gaming_hours                1000 non-null   float64
 4   game_genre                        1000 non-null   object 
 5   primary_game                      1000 non-null   object 
 6   gaming_platform                   1000 non-null   object 
 7   sleep_hours                       1000 non-null   float64
 8   sleep_quality                     1000 non-null   object 
 9   sleep_disruption_frequency        1000 non-null   object 
 10  academic_work_performance         1000 non-null   object 
 11  grades_gpa                        754 non-null    float64
 12  work_pr

### Data Preparation

In [639]:
df["gaming_addiction_risk_level"].unique()

array(['Severe', 'Low', 'High', 'Moderate'], dtype=object)

In [640]:
df["record_id"].nunique()

1000

In [641]:
df = df.drop(columns=["record_id"])

In [642]:
columns_to_hot_encode = [
    "gender",
    "game_genre", 
    "primary_game", 
    "gaming_platform",
    "mood_state",
]

df = pd.get_dummies(data=df, columns=columns_to_hot_encode)

In [643]:
sleep_quality_map = {
    "Insomnia": 0,
    "Very Poor": 1,
    "Poor": 2,
    "Fair": 3,
    "Good": 4
}

df["sleep_quality"] = df["sleep_quality"].map(arg=sleep_quality_map)

In [644]:
df["mood_swing_frequency"].unique()

array(['Never', 'Often', 'Rarely', 'Daily', 'Sometimes'], dtype=object)

In [645]:
mood_swing_freq_map = {
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Often": 3,
    "Daily": 4
}

df["mood_swing_frequency"] = df["mood_swing_frequency"].map(mood_swing_freq_map)

In [646]:
academic_perf_map = {
    "Failing": 0,
    "Poor": 1,
    "Below Average": 2,
    "Average": 3,
    "Good": 4,
    "Excellent": 5
}

df["academic_work_performance"] = df["academic_work_performance"].map(academic_perf_map)

In [647]:
df["sleep_disruption_frequency"].unique()

array(['Sometimes', 'Rarely', 'Often', 'Never', 'Always'], dtype=object)

In [648]:
sleep_disruption_map = {
    "Never": 0,
    "Rarely": 1,
    "Sometimes": 2,
    "Often": 3,
    "Always": 4
}

df["sleep_disruption_frequency"] = df["sleep_disruption_frequency"].map(sleep_disruption_map)

In [649]:
boolean_columns = df.select_dtypes(include="bool").columns
df[boolean_columns] = df[boolean_columns].astype(int)

In [650]:
addiction_map = {
    "Low": 0,
    "Moderate": 1,
    "High": 2,
    "Severe": 3
}

df["gaming_addiction_risk_level"] = df["gaming_addiction_risk_level"].map(addiction_map)

In [651]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 68 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   age                                1000 non-null   int64  
 1   daily_gaming_hours                 1000 non-null   float64
 2   sleep_hours                        1000 non-null   float64
 3   sleep_quality                      1000 non-null   int64  
 4   sleep_disruption_frequency         1000 non-null   int64  
 5   academic_work_performance          1000 non-null   int64  
 6   grades_gpa                         754 non-null    float64
 7   work_productivity_score            674 non-null    float64
 8   mood_swing_frequency               1000 non-null   int64  
 9   withdrawal_symptoms                1000 non-null   int64  
 10  loss_of_other_interests            1000 non-null   int64  
 11  continued_despite_problems         1000 non-null   int64 

#### Handling NULL Values

In [652]:
# NOTE: TEMPORARY
df["grades_gpa"] = df["grades_gpa"].fillna(df["grades_gpa"].median())
df["work_productivity_score"] = (
    df["work_productivity_score"]
    .fillna(df["work_productivity_score"].median())
)

#### Splitting and Scaling

In [653]:
y = df["gaming_addiction_risk_level"]
X = df.drop(columns=["gaming_addiction_risk_level"])

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
) 

In [654]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Logistic Regression

#### Simple Logistic Regression

In [655]:
logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0
)

logistic_regression.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.7
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [656]:
y_pred = logistic_regression.predict(X_test_scaled)

# Evaluation  
logistic_regression_accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", logistic_regression_accuracy) 

Accuracy: 0.92


In [657]:
coefficients = logistic_regression.coef_[0]
odds_ratios = np.exp(coefficients)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': coefficients,
    'odds_ratio': odds_ratios
})

feature_importance.sort_values(by='coefficient', ascending=False)

,feature,coefficient,odds_ratio
17,face_to_face_social_hours_weekly,0.840375,2.317236
2,sleep_hours,0.690322,1.994357
15,exercise_hours_weekly,0.409861,1.506608
3,sleep_quality,0.208737,1.232121
48,primary_game_PUBG Mobile,0.180558,1.197886
...,...,...,...
18,monthly_game_spending_usd,-1.209464,0.298357
1,daily_gaming_hours,-1.419983,0.241718
11,continued_despite_problems,-1.584647,0.205020
9,withdrawal_symptoms,-3.024024,0.048605


In [658]:
treshold = 0.2

feature_importance = feature_importance[abs(feature_importance["coefficient"]) > treshold]

feature_importance.sort_values(by='coefficient', ascending=False)
important_columns = feature_importance["feature"]

In [659]:
y = df["gaming_addiction_risk_level"]
X = df.drop(columns=["gaming_addiction_risk_level"])

X = X.loc[:, important_columns]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
) 

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [660]:
logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0
)

logistic_regression.fit(X_train_scaled, y_train)

y_pred = logistic_regression.predict(X_test_scaled)

# Evaluation  
logistic_regression_accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", logistic_regression_accuracy) 

Accuracy: 0.97


#### Cross Validation with Logistic Regression

In [661]:
y = df["gaming_addiction_risk_level"]
X = df.drop(columns=["gaming_addiction_risk_level"])

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=0.2, random_state=RANDOM_STATE  
) 

X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [664]:
scoring = ["accuracy", "f1_weighted", "precision_weighted", "recall_weighted"]

logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0
)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(logistic_regression, X_scaled, y, cv=cv, scoring="f1_weighted")

print("Fold accuracies:", scores)
print("Mean F1 Weighted:", np.mean(scores))

Fold accuracies: [0.93910714 0.97957521 0.95042184 0.96088204 0.96       0.96054586
 0.95963801 0.98       0.93982906 0.96040504]
Mean F1 Weighted: 0.959040420181547


In [666]:
results = cross_validate(
    logistic_regression,
    X_scaled,
    y,
    cv=cv,
    scoring=scoring
)

for metric in scoring:
    print(metric, results["test_" + metric].mean())

accuracy 0.959
f1_weighted 0.959040420181547
precision_weighted 0.9610947447805032
recall_weighted 0.959
